# DEC vs k-means на Mall Customers

Сравнение классических k-средних и глубокой встроенной кластеризации (DEC) в файле Mall_Customers.csv. Блокнот включает в себя предварительную обработку, обучение модели для обоих подходов, оценку на основе показателей и двухфункциональную визуализацию для параллельного анализа.

## Библиотеки и воспроизводимость

Импортируются необходимые библиотеки и задаются случайные начальные значения для Python, NumPy и TensorFlow. Он также обнаруживает доступное оборудование (ЦП/ГП), чтобы обеспечить единообразие выполнения в разных средах.

In [ ]:
import os
import re
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Enable deterministic ops when supported.
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

gpus = tf.config.list_physical_devices('GPU')
device_name = 'GPU' if gpus else 'CPU'
print(f'Active device: {device_name}')
print(f'GPU count: {len(gpus)}')

TIMINGS = {}

# Training settings for deeper, more stable evaluation.
TRAIN_CFG = {
    'ae_epochs': 150,
    'ae_patience': 15,
    'ae_lr': 1e-3,
    'dec_epochs': 300,
    'dec_lr': 5e-4,
    'dec_tol': 0.002,
    'update_interval': 5,
    'n_seeds_eval': 5,
}
print('Training config:', TRAIN_CFG)


## Загрузка набора данных

Загружается файл Mall_Customers.csv и печатаются образцы строк и форма набора данных для быстрой проверки.

In [ ]:
t0 = time.perf_counter()

DATA_PATH = 'Mall_Customers.csv'

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print('Mall_Customers.csv was not found. Please upload the dataset file:')
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise FileNotFoundError('No file was uploaded.')
        if DATA_PATH not in uploaded:
            DATA_PATH = list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError('Could not find or upload Mall_Customers.csv') from e

df = pd.read_csv(DATA_PATH)
print('Dataset path:', DATA_PATH)
print('Shape:', df.shape)
display(df.head())

TIMINGS['load_data_sec'] = time.perf_counter() - t0

## Очистка данных и подготовка функций


In [ ]:
t0 = time.perf_counter()

df_model = df.copy()

def normalize_col(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', name.lower())

normalized_map = {normalize_col(c): c for c in df_model.columns}

def find_col_by_keywords(keywords):
    for norm, orig in normalized_map.items():
        if all(k in norm for k in keywords):
            return orig
    return None

# Typical columns in Mall Customers dataset.
customer_id_col = find_col_by_keywords(['customer', 'id'])
if customer_id_col is None:
    customer_id_col = find_col_by_keywords(['id'])

gender_col = find_col_by_keywords(['gender'])

if customer_id_col in df_model.columns:
    df_model = df_model.drop(columns=[customer_id_col])
    print(f'Removed ID column: {customer_id_col}')

if gender_col is not None and gender_col in df_model.columns:
    g = df_model[gender_col].astype(str).str.strip().str.lower()
    mapped = g.map({'male': 1, 'm': 1, 'female': 0, 'f': 0})
    if mapped.isna().any():
        mapped = pd.factorize(g)[0]
    df_model[gender_col] = mapped.astype(float)
    print(f'Encoded gender column: {gender_col}')

print('Final columns:', list(df_model.columns))
print('Missing values per column:')
print(df_model.isna().sum())

display(df_model.describe(include='all'))

TIMINGS['cleaning_sec'] = time.perf_counter() - t0

## Матрица функций и масштабирование

Строится матрица объектов «X» из числовых столбцов, при необходимости вводятся недостающие значения и применяется «StandardScaler», чтобы методы, основанные на расстоянии, обрабатывали объекты в сопоставимых масштабах.

In [ ]:
t0 = time.perf_counter()

feature_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
if len(feature_cols) < 2:
    raise ValueError('Not enough numeric features for clustering.')

X_df = df_model[feature_cols].copy()
if X_df.isna().any().any():
    X_df = X_df.fillna(X_df.median(numeric_only=True))

scaler = StandardScaler()
X = scaler.fit_transform(X_df.values)

print('Used features:', feature_cols)
print('X shape:', X.shape)

TIMINGS['preprocess_sec'] = time.perf_counter() - t0

## Базовый уровень: k-средние
Выполняется быстрый поиск по силуэтам по `k=2..10`, выбирает лучший `k`, обучает окончательные `k-средние` и сохраняет метки базового кластера.

In [ ]:
t0 = time.perf_counter()

candidate_k = range(2, 11)
sil_scores = []
for k in candidate_k:
    km_tmp = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    labels_tmp = km_tmp.fit_predict(X)
    sil = silhouette_score(X, labels_tmp)
    sil_scores.append((k, sil))

best_k, best_sil = max(sil_scores, key=lambda t: t[1])
print('Silhouette by k:', sil_scores)
print(f'Selected k = {best_k} (best silhouette={best_sil:.4f})')

kmeans = KMeans(n_clusters=best_k, random_state=SEED, n_init=50)
labels_kmeans = kmeans.fit_predict(X)

TIMINGS['kmeans_sec'] = time.perf_counter() - t0

## Предварительное обучение автоэнкодера

Определяется компактный автокодировщик MLP и предварительно обучается его с ранней остановкой для получения стабильных скрытых представлений

In [ ]:
t0 = time.perf_counter()

X_train, X_val = train_test_split(X, test_size=0.2, random_state=SEED)

input_dim = X.shape[1]
latent_dim = int(np.clip(input_dim * 2, 8, 16))

inputs = layers.Input(shape=(input_dim,), name='input')
x = layers.Dense(32, activation='relu')(inputs)
x = layers.Dense(16, activation='relu')(x)
latent = layers.Dense(latent_dim, activation=None, name='latent')(x)

x = layers.Dense(16, activation='relu')(latent)
x = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(input_dim, activation='linear', name='reconstruction')(x)

autoencoder = Model(inputs=inputs, outputs=outputs, name='autoencoder')
encoder = Model(inputs=inputs, outputs=latent, name='encoder')

autoencoder.compile(optimizer=tf.keras.optimizers.Adam(TRAIN_CFG['ae_lr']), loss='mse')

batch_size = 128 if X.shape[0] >= 128 else max(16, X.shape[0] // 2)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=TRAIN_CFG['ae_patience'],
    restore_best_weights=True,
    min_delta=1e-4
)

history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=TRAIN_CFG['ae_epochs'],
    batch_size=batch_size,
    callbacks=[early_stop],
    verbose=0
)

print(f'AE pretrain epochs: {len(history.history["loss"])}')
print(f'Best val_loss: {min(history.history["val_loss"]):.6f}')

TIMINGS['ae_pretrain_sec'] = time.perf_counter() - t0


## Инициализация кластера скрытого пространства

Вычисляются скрытые вложения `Z = encoder(X)` и запускаются `k-средние` в скрытом пространстве для инициализации центров кластеров DEC.

In [ ]:
Z_init = encoder.predict(X, batch_size=256, verbose=0)

kmeans_z = KMeans(n_clusters=best_k, random_state=SEED, n_init=30)
labels_z_init = kmeans_z.fit_predict(Z_init)
cluster_centers_init = kmeans_z.cluster_centers_.astype('float32')

print('Latent Z shape:', Z_init.shape)
print('DEC center shape:', cluster_centers_init.shape)

## Точная настройка DEC (KL(P || Q))

Выполняется оптимизация DEC с мягкими назначениями Student-t `q`, целевым распределением `p` и минимизацией расхождения KL `KL(P || Q)` с периодическими целевыми обновлениями и ранней остановкой по стабильности метки.

In [ ]:
t0 = time.perf_counter()

class DECModel(tf.keras.Model):
    def __init__(self, encoder_model, n_clusters, alpha=1.0, init_centers=None):
        super().__init__()
        self.encoder = encoder_model
        self.n_clusters = n_clusters
        self.alpha = alpha
        if init_centers is None:
            raise ValueError('init_centers must be provided for stable DEC initialization.')
        self.cluster_centers = self.add_weight(
            shape=(n_clusters, init_centers.shape[1]),
            initializer=tf.keras.initializers.Constant(init_centers),
            trainable=True,
            name='cluster_centers'
        )

    def call(self, x, training=False):
        z = self.encoder(x, training=training)
        # Student's t-distribution for soft assignment q_ij.
        distances = tf.reduce_sum(tf.square(tf.expand_dims(z, axis=1) - self.cluster_centers), axis=2)
        q = 1.0 / (1.0 + distances / self.alpha)
        q = tf.pow(q, (self.alpha + 1.0) / 2.0)
        q = q / tf.reduce_sum(q, axis=1, keepdims=True)
        return q

def target_distribution(q_np):
    weight = (q_np ** 2) / np.clip(q_np.sum(axis=0, keepdims=True), 1e-8, None)
    return (weight.T / np.clip(weight.sum(axis=1), 1e-8, None)).T

X32 = X.astype('float32')

dec_model = DECModel(
    encoder_model=encoder,
    n_clusters=best_k,
    alpha=1.0,
    init_centers=cluster_centers_init
)

optimizer = tf.keras.optimizers.Adam(learning_rate=TRAIN_CFG['dec_lr'])

max_dec_epochs = TRAIN_CFG['dec_epochs']
update_interval = TRAIN_CFG['update_interval']
tol = TRAIN_CFG['dec_tol']

prev_labels = None
loss_history = []

for epoch in range(max_dec_epochs):
    if epoch % update_interval == 0:
        q_all = dec_model(X32, training=False).numpy()
        p_all = target_distribution(q_all).astype('float32')

        current_labels = q_all.argmax(axis=1)
        if prev_labels is None:
            delta = 1.0
        else:
            delta = np.mean(current_labels != prev_labels)

        prev_labels = current_labels.copy()
        if epoch > 0 and delta < tol:
            print(f'DEC early stop: epoch={epoch}, delta_label={delta:.6f}')
            break

    idx = np.random.permutation(X32.shape[0])
    epoch_losses = []

    for start in range(0, X32.shape[0], batch_size):
        batch_idx = idx[start:start + batch_size]
        xb = tf.convert_to_tensor(X32[batch_idx])
        pb = tf.convert_to_tensor(p_all[batch_idx])

        with tf.GradientTape() as tape:
            qb = dec_model(xb, training=True)
            # KL(P || Q)
            kl = tf.reduce_mean(tf.reduce_sum(pb * tf.math.log((pb + 1e-8) / (qb + 1e-8)), axis=1))

        grads = tape.gradient(kl, dec_model.trainable_weights)
        optimizer.apply_gradients(zip(grads, dec_model.trainable_weights))
        epoch_losses.append(float(kl.numpy()))

    mean_epoch_loss = float(np.mean(epoch_losses))
    loss_history.append(mean_epoch_loss)

    if epoch % 10 == 0:
        print(f'epoch={epoch:03d}, kl_loss={mean_epoch_loss:.6f}, delta_label={delta:.6f}')

q_final = dec_model(X32, training=False).numpy()
labels_dec = q_final.argmax(axis=1)
Z_dec = encoder.predict(X, batch_size=256, verbose=0)

TIMINGS['dec_finetune_sec'] = time.perf_counter() - t0


## Метрики и сравнение методов

Вычисляются «Силуэт», «Дэвис-Булден» и «Калински-Харабаш» в согласованном пространстве «X», а затем суммируются результаты для «k-средних» и DEC в одной таблице.

In [ ]:
def compute_metrics(X_space, labels):
    labels = np.asarray(labels)
    n_clusters_local = len(np.unique(labels))
    if n_clusters_local < 2:
        return {
            'silhouette': np.nan,
            'davies_bouldin': np.nan,
            'calinski_harabasz': np.nan
        }
    return {
        'silhouette': silhouette_score(X_space, labels),
        'davies_bouldin': davies_bouldin_score(X_space, labels),
        'calinski_harabasz': calinski_harabasz_score(X_space, labels)
    }

metrics_km = compute_metrics(X, labels_kmeans)
metrics_dec = compute_metrics(X, labels_dec)

results = pd.DataFrame([
    {
        'Method': 'k-means',
        'Silhouette (X)': metrics_km['silhouette'],
        'Davies-Bouldin (X)': metrics_km['davies_bouldin'],
        'Calinski-Harabasz (X)': metrics_km['calinski_harabasz'],
        'Inertia (aux)': kmeans.inertia_
    },
    {
        'Method': 'DEC',
        'Silhouette (X)': metrics_dec['silhouette'],
        'Davies-Bouldin (X)': metrics_dec['davies_bouldin'],
        'Calinski-Harabasz (X)': metrics_dec['calinski_harabasz'],
        'Inertia (aux)': np.nan
    }
## Оценка стабильности

В этом разделе повторяются k-средние и DEC для нескольких случайных начальных значений, а метрики сообщаются как среднее +/- стандартное. Это дает более надежное сравнение, чем одиночный прогон.

In [ ]:
t0 = time.perf_counter()

def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def build_autoencoder_models(input_dim, latent_dim):
    inp = layers.Input(shape=(input_dim,), name='input')
    h = layers.Dense(32, activation='relu')(inp)
    h = layers.Dense(16, activation='relu')(h)
    z = layers.Dense(latent_dim, activation=None, name='latent')(h)

    h = layers.Dense(16, activation='relu')(z)
    h = layers.Dense(32, activation='relu')(h)
    out = layers.Dense(input_dim, activation='linear', name='reconstruction')(h)

    ae = Model(inputs=inp, outputs=out, name='autoencoder')
    enc = Model(inputs=inp, outputs=z, name='encoder')
    return ae, enc

seed_list = [SEED + 10 * i for i in range(TRAIN_CFG['n_seeds_eval'])]
rows = []

for run_seed in seed_list:
    tf.keras.backend.clear_session()
    set_all_seeds(run_seed)

    batch_size_local = 128 if X.shape[0] >= 128 else max(16, X.shape[0] // 2)

    km_local = KMeans(n_clusters=best_k, random_state=run_seed, n_init=50)
    labels_km_local = km_local.fit_predict(X)

    X_train_local, X_val_local = train_test_split(X, test_size=0.2, random_state=run_seed)

    ae_local, enc_local = build_autoencoder_models(input_dim=X.shape[1], latent_dim=latent_dim)
    ae_local.compile(optimizer=tf.keras.optimizers.Adam(TRAIN_CFG['ae_lr']), loss='mse')

    es_local = EarlyStopping(
        monitor='val_loss',
        patience=TRAIN_CFG['ae_patience'],
        restore_best_weights=True,
        min_delta=1e-4
    )

    ae_local.fit(
        X_train_local, X_train_local,
        validation_data=(X_val_local, X_val_local),
        epochs=TRAIN_CFG['ae_epochs'],
        batch_size=batch_size_local,
        callbacks=[es_local],
        verbose=0
    )

    Z_local = enc_local.predict(X, batch_size=256, verbose=0)
    km_z_local = KMeans(n_clusters=best_k, random_state=run_seed, n_init=30)
    centers_local = km_z_local.fit(Z_local).cluster_centers_.astype('float32')

    dec_local = DECModel(
        encoder_model=enc_local,
        n_clusters=best_k,
        alpha=1.0,
        init_centers=centers_local
    )
    opt_local = tf.keras.optimizers.Adam(learning_rate=TRAIN_CFG['dec_lr'])

    X32_local = X.astype('float32')
    prev_labels_local = None

    for epoch in range(TRAIN_CFG['dec_epochs']):
        if epoch % TRAIN_CFG['update_interval'] == 0:
            q_all_local = dec_local(X32_local, training=False).numpy()
            p_all_local = target_distribution(q_all_local).astype('float32')

            labels_now = q_all_local.argmax(axis=1)
            if prev_labels_local is None:
                delta_local = 1.0
            else:
                delta_local = np.mean(labels_now != prev_labels_local)

            prev_labels_local = labels_now.copy()
            if epoch > 0 and delta_local < TRAIN_CFG['dec_tol']:
                break

        idx_local = np.random.permutation(X32_local.shape[0])
        for start in range(0, X32_local.shape[0], batch_size_local):
            batch_idx_local = idx_local[start:start + batch_size_local]
            xb_local = tf.convert_to_tensor(X32_local[batch_idx_local])
            pb_local = tf.convert_to_tensor(p_all_local[batch_idx_local])

            with tf.GradientTape() as tape:
                qb_local = dec_local(xb_local, training=True)
                kl_local = tf.reduce_mean(
                    tf.reduce_sum(pb_local * tf.math.log((pb_local + 1e-8) / (qb_local + 1e-8)), axis=1)
                )

            grads_local = tape.gradient(kl_local, dec_local.trainable_weights)
            opt_local.apply_gradients(zip(grads_local, dec_local.trainable_weights))

    labels_dec_local = dec_local(X32_local, training=False).numpy().argmax(axis=1)

    m_km_local = compute_metrics(X, labels_km_local)
    m_dec_local = compute_metrics(X, labels_dec_local)

    rows.append({
        'Seed': run_seed,
        'Method': 'k-means',
        'Silhouette (X)': m_km_local['silhouette'],
        'Davies-Bouldin (X)': m_km_local['davies_bouldin'],
        'Calinski-Harabasz (X)': m_km_local['calinski_harabasz']
    })
    rows.append({
        'Seed': run_seed,
        'Method': 'DEC',
        'Silhouette (X)': m_dec_local['silhouette'],
        'Davies-Bouldin (X)': m_dec_local['davies_bouldin'],
        'Calinski-Harabasz (X)': m_dec_local['calinski_harabasz']
    })

    print(f'seed={run_seed}: completed')

multi_seed_results = pd.DataFrame(rows)
display(multi_seed_results)

summary = multi_seed_results.groupby('Method')[
    ['Silhouette (X)', 'Davies-Bouldin (X)', 'Calinski-Harabasz (X)']
].agg(['mean', 'std'])
summary.columns = [f'{col}_{stat}' for col, stat in summary.columns]
summary = summary.reset_index()

for metric in ['Silhouette (X)', 'Davies-Bouldin (X)', 'Calinski-Harabasz (X)']:
    summary[f'{metric} mean?std'] = summary.apply(
        lambda r: f"{r[f'{metric}_mean']:.4f} ? {r[f'{metric}_std']:.4f}", axis=1
    )

summary_view = summary[[
    'Method',
    'Silhouette (X) mean?std',
    'Davies-Bouldin (X) mean?std',
    'Calinski-Harabasz (X) mean?std'
]]

display(summary_view)

TIMINGS['multiseed_eval_sec'] = time.perf_counter() - t0


## Бюджет времени выполнения

Сообщается время для ключевых этапов (загрузка, предварительная обработка, k-средние, предварительное обучение автокодировщика, точная настройка DEC) для проверки соответствия выполнения свободным ограничениям Colab.

In [ ]:
timing_order = [
    'load_data_sec',
    'cleaning_sec',
    'preprocess_sec',
    'kmeans_sec',
    'ae_pretrain_sec',
    'dec_finetune_sec',
    'multiseed_eval_sec'
]

timing_rows = []
for key in timing_order:
    if key in TIMINGS:
        timing_rows.append({'Stage': key.replace('_sec', ''), 'Seconds': TIMINGS[key]})

timing_df = pd.DataFrame(timing_rows)
if not timing_df.empty:
    timing_df['Minutes'] = timing_df['Seconds'] / 60.0
    display(timing_df)
    total_sec = timing_df['Seconds'].sum()
    print(f'Total measured time: {total_sec:.2f} sec ({total_sec / 60:.2f} min)')

print('Estimated Colab Free budget with deeper training (typical):')
print('- Preprocessing + k-means: < 1 min')
print('- AE pretraining (up to 150 epochs, EarlyStopping): ~2-8 min')
print('- DEC fine-tuning (up to 300 epochs, early stop): ~5-20 min')
print('- Multi-seed stability block (5 seeds): ~10-40 min (depends on early stop and device)')
print("- If runtime is too high, reduce TRAIN_CFG['n_seeds_eval'] from 5 to 3.")


## Кластерная визуализация двух функций

Строятся параллельные диаграммы рассеяния для «k-средних» и DEC с использованием одних и тех же двух исходных функций и выровненных осей для прямого визуального сравнения.

In [ ]:
def pick_feature(preferred_keywords, available_cols):
    norm_available = {re.sub(r'[^a-z0-9]+', '', c.lower()): c for c in available_cols}
    for kw_group in preferred_keywords:
        for norm, orig in norm_available.items():
            if all(k in norm for k in kw_group):
                return orig
    return None

all_cols = list(X_df.columns)
feature_x = pick_feature([['annual', 'income'], ['income']], all_cols)
feature_y = pick_feature([['spending', 'score'], ['score'], ['spending']], all_cols)

if feature_x is None or feature_y is None or feature_x == feature_y:
    fallback = all_cols[:2]
    feature_x, feature_y = fallback[0], fallback[1]

print('Visualization features:', feature_x, '|', feature_y)

plot_df = df_model.copy()
plot_df['cluster_kmeans'] = labels_kmeans
plot_df['cluster_dec'] = labels_dec

x_vals = plot_df[feature_x].values
y_vals = plot_df[feature_y].values

xmin, xmax = x_vals.min(), x_vals.max()
ymin, ymax = y_vals.min(), y_vals.max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

axes[0].scatter(plot_df[feature_x], plot_df[feature_y], c=plot_df['cluster_kmeans'], cmap='tab10', s=45, alpha=0.85)
axes[0].set_title('k-means clustering')
axes[0].set_xlabel(feature_x)
axes[0].set_ylabel(feature_y)
axes[0].set_xlim(xmin - 1e-9, xmax + 1e-9)
axes[0].set_ylim(ymin - 1e-9, ymax + 1e-9)

axes[1].scatter(plot_df[feature_x], plot_df[feature_y], c=plot_df['cluster_dec'], cmap='tab10', s=45, alpha=0.85)
axes[1].set_title('DEC clustering')
axes[1].set_xlabel(feature_x)
axes[1].set_ylabel(feature_y)
axes[1].set_xlim(xmin - 1e-9, xmax + 1e-9)
axes[1].set_ylim(ymin - 1e-9, ymax + 1e-9)

plt.suptitle(f'Cluster comparison on features: {feature_x} vs {feature_y}', y=1.02)
plt.tight_layout()
plt.show()

### Итог:
**На Mall_Customers лучше сработал k-means.**

k-means > DEC по всем метрикам:

- Silhouette: 0.4208 vs 0.3992
- Davies-Bouldin: 0.8331 vs 0.8806 (меньше лучше)
- Calinski-Harabasz: 89.98 vs 79.26

По 5 seed результат устойчив: k-means стабильно лучше, DEC более вариативен.
Вычисления без ручной загрузки данных занимают около 1.5 мин.